In [ ]:
import scanpy as sc
import matplotlib
from matplotlib import rcParams

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, dpi_save=300, color_map='Reds', facecolor='white')
matplotlib.rcParams.update({'figure.figsize': (6, 6)})


In [ ]:
adata = sc.read_h5ad('/mnt/yiming/nfs_share/hema_wave/hemo-bgi-d1-10-bbknn-annotated.h5ad')

In [ ]:
# Filter data
adata_endo = adata[adata.obs['CellType_man'] == 'Endo'].copy()
sc.pp.normalize_total(adata_endo)
sc.pp.log1p(adata_endo)
sc.pp.highly_variable_genes(adata_endo, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pp.scale(adata_endo)
sc.tl.pca(adata_endo)
sc.pp.neighbors(adata_endo)
sc.tl.umap(adata_endo)
sc.tl.leiden(adata)

In [ ]:
sc.pl.dotplot(
    adata_endo,
    ['POU5F1', 'SOX17', 'FOXA2', 'CXCR4', 
     'GATA4', 'GATA6', 'PDGFRA', 
     'AFP', 'APOA1', 'CDX2', 'HAND1', 
     'HBZ', 'HBE1', 'GYPA', 'GYPB'],
    groupby='subcluster_man',
    dendrogram=False,  
    cmap='Reds',
    # var_group_rotation=90,
    standard_scale='var',
    swap_axes=True,
    figsize=(3, 5), 
    # title='HCBE Endoderm subtypes marker gene expression',
    show=False
)
plt.savefig(f'/mnt/yiming/nfs_share/hema_wave/2512-revision/2_hemo10_dotplot_endo_geneexp.svg', 
                format='svg', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(adata_endo, color=['subcluster_man'], 
           legend_loc='on data', legend_fontsize=15, legend_fontoutline=2, 
           title='HCEB Endo Subtype',
          frameon=False, show=False)
plt.savefig(f'/mnt/yiming/nfs_share/hema_wave/2512-revision/2_hemo10_umap_endo_subtype_legendon.svg', 
                format='svg', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
cluster_annotations = {
    '0': 'Early Hypo',
    '1': 'CDX2+ Hypo',
    '2': 'YS.Endo',        
    '3': 'Late Hypo'}

adata_endo.obs['subcluster_man'] = adata_endo.obs['leiden'].map(cluster_annotations)
adata.write('./2512-revision/hemo10_endo.h5ad')

In [ ]:
# Define genes
genes = ['HBZ', 'HBE1', 'HBG1', 'GYPA']
colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3']

# Define genes
genes = ['SOX17', 'GATA6', 'FOXA2', 'CDX2']
colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3']

# Create figure
fig, ax = plt.subplots(figsize=(6, 3))

# Plot each gene
for gene, color in zip(genes, colors):
    if gene in adata_endo.var_names:
        # Get expression data
        expr = adata_endo[:, gene].X
        if hasattr(expr, 'toarray'):
            expr = expr.toarray().flatten()
        else:
            expr = expr.flatten()
        
        time_points = adata_endo.obs['day'].values
        
        # Calculate mean expression per time point
        unique_times = np.unique(time_points)
        mean_expr = []
        mean_time = []
        
        for t in unique_times:
            mask = time_points == t
            if np.sum(mask) > 0:
                mean_expr.append(np.mean(expr[mask]))
                mean_time.append(t)
        
        mean_expr = np.array(mean_expr)
        mean_time = np.array(mean_time)
        
        # Sort by time
        sort_idx = np.argsort(mean_time)
        mean_time_sorted = mean_time[sort_idx]
        mean_expr_sorted = mean_expr[sort_idx]
        
        # Apply Gaussian smoothing
        sigma = 1.0
        expr_smoothed = gaussian_filter1d(mean_expr_sorted, sigma=sigma)
        
        # Plot smoothed line
        ax.plot(mean_time_sorted, expr_smoothed, 
                color=color, 
                linewidth=2.5,
                label=gene,
                zorder=2)

# Labels
ax.set_xlabel('Sample day')
ax.set_ylabel('Expression')
ax.set_title('Gene expression of HCEB Endo')

# Grid and legend
ax.grid(True)
ax.legend(frameon=True)

# Set y-axis to start at 0
ax.set_ylim(bottom=0)

# Layout
plt.tight_layout()
plt.savefig(f'/mnt/yiming/nfs_share/hema_wave/2512-revision/2_hemo10_line_endo_erygene-2.svg', 
                format='svg', dpi=300, bbox_inches='tight')
plt.show()
plt.close()